[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_81_Reliability_and_Partial_Failure.ipynb)

# Lesson 81 — Reliability & Partial Failure
### Phase 9 · Advanced Multi-Agent Orchestration · Lesson 5 of ~6

In **Lesson 80** you built a **Planner**: it decomposes a goal into an ordered
DAG of sub-tasks, dispatches each through the L79 **Router**, threads each
result into the tasks that depend on it, and — when a step's answer failed a
validity check — did a **single-shot replan** onto the generalist.

That single-shot replan is a *hint* of reliability, not reliability itself.
A production orchestrator faces failures the L80 Planner has no answer for:

| Failure in the wild | L80 Planner's behaviour | What it needs |
|---|---|---|
| A step's backend raises a **transient** error (503, rate-limit) | the exception propagates → the whole `execute()` **crashes** | **retry** the blip |
| A step retries forever against a struggling service | — | **backoff + jitter** so you neither give up early nor stampede |
| A step **hangs** | the plan stalls indefinitely | a **timeout / deadline** |
| One dependency is **persistently down** | every step keeps calling it | a **circuit breaker** to isolate it |
| **2 of 5** parallel steps fail | one exception kills the layer | a **partial-failure policy** |

This lesson turns L80's one-line replan into a real **reliability layer** — the
discipline that makes orchestration *production-trustworthy*, and the last
building block before the **L82 Phase-9 capstone** (shipping `orchestra` as your
4th open-source artifact).

**Roadmap:**

| Lesson | Topic | Ships |
|---|---|---|
| L77 | Multi-agent topologies | `orchestra/core.py` |
| L78 | Shared state & the blackboard | `orchestra/blackboard.py` |
| L79 | Routing & handoff | `orchestra/router.py` |
| L80 | Planning & decomposition | `orchestra/planner.py` |
| **L81 (today)** | **Reliability & partial failure** | **`orchestra/reliability.py`** |
| L82 | **Phase-9 capstone** | ship `orchestra` (4th OSS artifact) |


In [ ]:
# ── Setup: write the orchestra package to disk, then import it ──────────────
# On Google Colab BASE = "/content" (its working dir). The four modules are
# embedded as base64 so no quote or docstring inside them can ever collide with
# a notebook cell delimiter (the pattern we settled on back in L77).
import base64, os, sys, importlib

BASE = "/content"          # Colab's working directory
os.makedirs(os.path.join(BASE, "orchestra"), exist_ok=True)

_MODULES = {
    "core.py":        "IyBvcmNoZXN0cmEvY29yZS5weSAgLS0gIG1lc3NhZ2UgKyBhZ2VudCBwcmltaXRpdmVzIChmcm9tIExlc3NvbiA3NykKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBDYWxsYWJsZQoKQGRhdGFjbGFzcwpjbGFzcyBNZXNzYWdlOgogICAgc2VuZGVyOiBzdHIKICAgIHJlY2lwaWVudDogc3RyCiAgICBraW5kOiBzdHIgICAgICAgICAgICAgICAgICMgInRhc2siIHwgInJlc3VsdCIgfCAiaGFuZG9mZiIgfCAuLi4KICAgIGNvbnRlbnQ6IEFueQogICAgbWV0YTogZGljdCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKY2xhc3MgQWdlbnQ6CiAgICAjIEFuIGFnZW50ID0gb25lIGNhbGxhYmxlICJicmFpbiIgYmVoaW5kIGEgbmFtZS4gYmFja2VuZChuYW1lLCBjb250ZW50KSAtPiAob3V0cHV0LCB0b2tlbnMpCiAgICBkZWYgX19pbml0X18oc2VsZiwgbmFtZTogc3RyLCByb2xlOiBzdHIsIGJhY2tlbmQ6IENhbGxhYmxlKToKICAgICAgICBzZWxmLm5hbWUgPSBuYW1lCiAgICAgICAgc2VsZi5yb2xlID0gcm9sZQogICAgICAgIHNlbGYuYmFja2VuZCA9IGJhY2tlbmQKICAgICAgICBzZWxmLmNhbGxzID0gMAogICAgZGVmIGFjdChzZWxmLCBtc2c6ICJNZXNzYWdlIikgLT4gIk1lc3NhZ2UiOgogICAgICAgIHNlbGYuY2FsbHMgKz0gMQogICAgICAgIG91dCwgdG9rZW5zID0gc2VsZi5iYWNrZW5kKHNlbGYubmFtZSwgbXNnLmNvbnRlbnQpCiAgICAgICAgcmV0dXJuIE1lc3NhZ2Uoc2VuZGVyPXNlbGYubmFtZSwgcmVjaXBpZW50PW1zZy5zZW5kZXIsCiAgICAgICAgICAgICAgICAgICAgICAga2luZD0icmVzdWx0IiwgY29udGVudD1vdXQsIG1ldGE9eyJ0b2tlbnMiOiB0b2tlbnN9KQo=",
    "router.py":      "IyBvcmNoZXN0cmEvcm91dGVyLnB5ICAtLSAgYSByb3V0ZXIgdGhhdCBjbGFzc2lmaWVzIGVhY2ggdGFzayBhbmQgaGFuZHMgaXQKIyB0byB0aGUgYmVzdCBzcGVjaWFsaXN0LCBlc2NhbGF0ZXMgd2hlbiB1bnN1cmUsIGFuZCBzdXJ2aXZlcyBoYW5kb2ZmIGxvb3BzLgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsLCBEaWN0CmZyb20gb3JjaGVzdHJhLmNvcmUgaW1wb3J0IE1lc3NhZ2UKCkBkYXRhY2xhc3MKY2xhc3MgUm91dGU6CiAgICBjYXRlZ29yeTogT3B0aW9uYWxbc3RyXQogICAgY29uZmlkZW5jZTogZmxvYXQKCmNsYXNzIENsYXNzaWZpZXI6CiAgICAjIENoZWFwIGtleXdvcmQgY2xhc3NpZmllci4gSW4gcHJvZHVjdGlvbiB0aGlzIHdvdWxkIGJlIGFuIGVtYmVkZGluZyBtb2RlbAogICAgIyBvciBhIHNtYWxsIExMTTsgdGhlIGludGVyZmFjZSAodGV4dCAtPiBSb3V0ZSkgaXMgd2hhdCBtYXR0ZXJzLgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGtleXdvcmRzOiBEaWN0W3N0ciwgbGlzdF0pOgogICAgICAgIHNlbGYua2V5d29yZHMgPSBrZXl3b3JkcwogICAgICAgIHNlbGYuY2FsbHMgPSAwCiAgICBkZWYgY2xhc3NpZnkoc2VsZiwgdGV4dDogc3RyKSAtPiBSb3V0ZToKICAgICAgICBzZWxmLmNhbGxzICs9IDEKICAgICAgICB0ID0gdGV4dC5sb3dlcigpCiAgICAgICAgc2NvcmVzID0ge30KICAgICAgICBmb3IgY2F0LCBrd3MgaW4gc2VsZi5rZXl3b3Jkcy5pdGVtcygpOgogICAgICAgICAgICBoaXRzID0gc3VtKDEgZm9yIGsgaW4ga3dzIGlmIGsgaW4gdCkKICAgICAgICAgICAgaWYgaGl0czoKICAgICAgICAgICAgICAgIHNjb3Jlc1tjYXRdID0gaGl0cwogICAgICAgIGlmIG5vdCBzY29yZXM6CiAgICAgICAgICAgIHJldHVybiBSb3V0ZShOb25lLCAwLjApICAgICAgICAgICAgICAjIG5vdGhpbmcgbWF0Y2hlZCAtPiBlc2NhbGF0ZQogICAgICAgIHRvdGFsID0gc3VtKHNjb3Jlcy52YWx1ZXMoKSkKICAgICAgICBiZXN0X2NhdCwgYmVzdF9oaXRzID0gc29ydGVkKHNjb3Jlcy5pdGVtcygpLCBrZXk9bGFtYmRhIGt2OiAoLWt2WzFdLCBrdlswXSkpWzBdCiAgICAgICAgcmV0dXJuIFJvdXRlKGJlc3RfY2F0LCBiZXN0X2hpdHMgLyB0b3RhbCkKCmNsYXNzIFJvdXRlcjoKICAgICMgSG9sZHMgc3BlY2lhbGlzdHMgKGNhdGVnb3J5IC0+IEFnZW50KSwgYSBnZW5lcmFsaXN0IGZhbGxiYWNrLCBhbmQgdGhlCiAgICAjIHBvbGljeTogZ2F0ZSBvbiBjb25maWRlbmNlLCBkaXNwYXRjaCwgZm9sbG93IGhhbmRvZmZzLCBjYXAgdGhlIGhvcHMuCiAgICBkZWYgX19pbml0X18oc2VsZiwgY2xhc3NpZmllciwgc3BlY2lhbGlzdHMsIGdlbmVyYWxpc3QsCiAgICAgICAgICAgICAgICAgdGhyZXNob2xkPTAuNiwgaG9wX2NhcD0zLCBjbGFzc2lmeV9jb3N0PTIpOgogICAgICAgIHNlbGYuY2xhc3NpZmllciA9IGNsYXNzaWZpZXIKICAgICAgICBzZWxmLnNwZWNpYWxpc3RzID0gZGljdChzcGVjaWFsaXN0cykKICAgICAgICBzZWxmLmdlbmVyYWxpc3QgPSBnZW5lcmFsaXN0CiAgICAgICAgc2VsZi50aHJlc2hvbGQgPSB0aHJlc2hvbGQKICAgICAgICBzZWxmLmhvcF9jYXAgPSBob3BfY2FwCiAgICAgICAgc2VsZi5jbGFzc2lmeV9jb3N0ID0gY2xhc3NpZnlfY29zdAoKICAgIGRlZiBfZXNjYWxhdGUoc2VsZiwgdGFzaywgdHJhY2UsIHRva2VucywgcmVhc29uKToKICAgICAgICBob3BzID0gbGVuKHRyYWNlKQogICAgICAgIG0gPSBzZWxmLmdlbmVyYWxpc3QuYWN0KE1lc3NhZ2UoInJvdXRlciIsIHNlbGYuZ2VuZXJhbGlzdC5uYW1lLCAidGFzayIsIHRhc2spKQogICAgICAgIHRyYWNlLmFwcGVuZChzZWxmLmdlbmVyYWxpc3QubmFtZSkKICAgICAgICB0b2tlbnMgKz0gbS5tZXRhLmdldCgidG9rZW5zIiwgMCkKICAgICAgICByZXR1cm4geyJhbnN3ZXIiOiBtLmNvbnRlbnQuZ2V0KCJhbnN3ZXIiKSwgInRva2VucyI6IHRva2VucywgInRyYWNlIjogdHJhY2UsCiAgICAgICAgICAgICAgICAiZXNjYWxhdGVkIjogVHJ1ZSwgInJlYXNvbiI6IHJlYXNvbiwgImhvcHMiOiBob3BzfQoKICAgIGRlZiBkaXNwYXRjaChzZWxmLCB0YXNrKToKICAgICAgICB0cmFjZSA9IFtdCiAgICAgICAgdG9rZW5zID0gc2VsZi5jbGFzc2lmeV9jb3N0CiAgICAgICAgcm91dGUgPSBzZWxmLmNsYXNzaWZpZXIuY2xhc3NpZnkodGFza1sidGV4dCJdKQogICAgICAgIGlmIHJvdXRlLmNhdGVnb3J5IGlzIE5vbmUgb3Igcm91dGUuY29uZmlkZW5jZSA8IHNlbGYudGhyZXNob2xkOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fZXNjYWxhdGUodGFzaywgdHJhY2UsIHRva2VucywgImxvd19jb25maWRlbmNlIikKICAgICAgICBjdXJyZW50ID0gcm91dGUuY2F0ZWdvcnkKICAgICAgICBob3BzID0gMAogICAgICAgIHdoaWxlIGhvcHMgPCBzZWxmLmhvcF9jYXA6CiAgICAgICAgICAgIGhvcHMgKz0gMQogICAgICAgICAgICBpZiBjdXJyZW50IG5vdCBpbiBzZWxmLnNwZWNpYWxpc3RzOgogICAgICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2VzY2FsYXRlKHRhc2ssIHRyYWNlLCB0b2tlbnMsICJ1bmtub3duX2NhdGVnb3J5IikKICAgICAgICAgICAgYWdlbnQgPSBzZWxmLnNwZWNpYWxpc3RzW2N1cnJlbnRdCiAgICAgICAgICAgIG0gPSBhZ2VudC5hY3QoTWVzc2FnZSgicm91dGVyIiwgYWdlbnQubmFtZSwgInRhc2siLCB0YXNrKSkKICAgICAgICAgICAgdHJhY2UuYXBwZW5kKGFnZW50Lm5hbWUpCiAgICAgICAgICAgIHRva2VucyArPSBtLm1ldGEuZ2V0KCJ0b2tlbnMiLCAwKQogICAgICAgICAgICB0YXJnZXQgPSBtLmNvbnRlbnQuZ2V0KCJoYW5kb2ZmIikKICAgICAgICAgICAgaWYgdGFyZ2V0IGlzIE5vbmU6CiAgICAgICAgICAgICAgICByZXR1cm4geyJhbnN3ZXIiOiBtLmNvbnRlbnQuZ2V0KCJhbnN3ZXIiKSwgInRva2VucyI6IHRva2VucywKICAgICAgICAgICAgICAgICAgICAgICAgInRyYWNlIjogdHJhY2UsICJlc2NhbGF0ZWQiOiBGYWxzZSwgInJlYXNvbiI6ICJyb3V0ZWQiLCAiaG9wcyI6IGhvcHN9CiAgICAgICAgICAgIGN1cnJlbnQgPSB0YXJnZXQKICAgICAgICByZXR1cm4gc2VsZi5fZXNjYWxhdGUodGFzaywgdHJhY2UsIHRva2VucywgImhvcF9jYXAiKQo=",
    "planner.py":     "IyBvcmNoZXN0cmEvcGxhbm5lci5weSAgLS0gIGRlY29tcG9zZSBhIGdvYWwgaW50byBhbiBvcmRlcmVkIHBsYW4gKGEgREFHIG9mCiMgc3ViLXRhc2tzKSwgZGlzcGF0Y2ggZWFjaCBzdWItdGFzayB0aHJvdWdoIHRoZSBMZXNzb24tNzkgUm91dGVyLCB0aHJlYWQgZWFjaAojIGRlcGVuZGVuY3kncyBvdXRwdXQgaW50byB0aGUgdGFza3MgdGhhdCBkZXBlbmQgb24gaXQsIGFuZCBhc3NlbWJsZSBhIHJlc3VsdC4KZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQ2FsbGFibGUsIE9wdGlvbmFsLCBMaXN0LCBEaWN0CmZyb20gb3JjaGVzdHJhLmNvcmUgaW1wb3J0IE1lc3NhZ2UKCkBkYXRhY2xhc3MKY2xhc3MgU3RlcDoKICAgIGlkOiBzdHIKICAgIHRleHQ6IHN0cgogICAgZGVwczogTGlzdFtzdHJdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpICAgIyBpZHMgdGhpcyBzdGVwIHdhaXRzIG9uCiAgICBwYXlsb2FkOiBkaWN0ID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpICAgICAgIyB0aGUgYXRvbWljIHRhc2sgZmllbGRzCgpAZGF0YWNsYXNzCmNsYXNzIFBsYW46CiAgICBnb2FsOiBzdHIKICAgIHN0ZXBzOiBMaXN0W1N0ZXBdCiAgICBkZWYgYnlfaWQoc2VsZikgLT4gRGljdFtzdHIsICJTdGVwIl06CiAgICAgICAgcmV0dXJuIHtzLmlkOiBzIGZvciBzIGluIHNlbGYuc3RlcHN9CgpkZWYgdG9wb19vcmRlcihzdGVwczogTGlzdFtTdGVwXSkgLT4gTGlzdFtzdHJdOgogICAgIyBLYWhuJ3MgYWxnb3JpdGhtLiBSYWlzZXMgVmFsdWVFcnJvciBvbiBhIG1pc3NpbmcgZGVwIG9yIGEgY3ljbGUuCiAgICBpZHMgPSB7cy5pZCBmb3IgcyBpbiBzdGVwc30KICAgIGluZGVnID0ge3MuaWQ6IDAgZm9yIHMgaW4gc3RlcHN9CiAgICBhZGogPSB7cy5pZDogW10gZm9yIHMgaW4gc3RlcHN9CiAgICBmb3IgcyBpbiBzdGVwczoKICAgICAgICBmb3IgZCBpbiBzLmRlcHM6CiAgICAgICAgICAgIGlmIGQgbm90IGluIGlkczoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInN0ZXAgJXIgZGVwZW5kcyBvbiB1bmtub3duIHN0ZXAgJXIiICUgKHMuaWQsIGQpKQogICAgICAgICAgICBhZGpbZF0uYXBwZW5kKHMuaWQpCiAgICAgICAgICAgIGluZGVnW3MuaWRdICs9IDEKICAgIHJlYWR5ID0gc29ydGVkKFtpIGZvciBpIGluIGluZGVnIGlmIGluZGVnW2ldID09IDBdKSAgICMgZGV0ZXJtaW5pc3RpYyBvcmRlcgogICAgb3JkZXIgPSBbXQogICAgd2hpbGUgcmVhZHk6CiAgICAgICAgbiA9IHJlYWR5LnBvcCgwKQogICAgICAgIG9yZGVyLmFwcGVuZChuKQogICAgICAgIGZvciBtIGluIGFkaltuXToKICAgICAgICAgICAgaW5kZWdbbV0gLT0gMQogICAgICAgICAgICBpZiBpbmRlZ1ttXSA9PSAwOgogICAgICAgICAgICAgICAgcmVhZHkuYXBwZW5kKG0pCiAgICAgICAgcmVhZHkuc29ydCgpCiAgICBpZiBsZW4ob3JkZXIpICE9IGxlbihzdGVwcyk6CiAgICAgICAgc3R1Y2sgPSBsZW4oc3RlcHMpIC0gbGVuKG9yZGVyKQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInBsYW4gaGFzIGEgY3ljbGU7ICVkIG9mICVkIHN0ZXBzIG5ldmVyIGJlY2FtZSByZWFkeSIKICAgICAgICAgICAgICAgICAgICAgICAgICUgKHN0dWNrLCBsZW4oc3RlcHMpKSkKICAgIHJldHVybiBvcmRlcgoKZGVmIGxheWVycyhzdGVwczogTGlzdFtTdGVwXSkgLT4gTGlzdFtMaXN0W3N0cl1dOgogICAgIyBHcm91cCBzdGVwcyBpbnRvIGRlcGVuZGVuY3kgbGF5ZXJzLiBFdmVyeSBzdGVwIGluIGEgbGF5ZXIgY2FuIHJ1biBpbgogICAgIyBwYXJhbGxlbDsgbGF5ZXIgayBkZXBlbmRzIG9ubHkgb24gbGF5ZXJzIDwgay4gVmFsaWRhdGVzIChyYWlzZXMgb24gY3ljbGUpLgogICAgb3JkZXIgPSB0b3BvX29yZGVyKHN0ZXBzKQogICAgYnlfaWQgPSB7cy5pZDogcyBmb3IgcyBpbiBzdGVwc30KICAgIGRlcHRoID0ge30KICAgIGZvciBzaWQgaW4gb3JkZXI6CiAgICAgICAgZHMgPSBieV9pZFtzaWRdLmRlcHMKICAgICAgICBkZXB0aFtzaWRdID0gMCBpZiBub3QgZHMgZWxzZSAxICsgbWF4KGRlcHRoW2RdIGZvciBkIGluIGRzKQogICAgb3V0ID0gW10KICAgIGZvciBzaWQgaW4gb3JkZXI6CiAgICAgICAgZCA9IGRlcHRoW3NpZF0KICAgICAgICB3aGlsZSBsZW4ob3V0KSA8PSBkOgogICAgICAgICAgICBvdXQuYXBwZW5kKFtdKQogICAgICAgIG91dFtkXS5hcHBlbmQoc2lkKQogICAgcmV0dXJuIFtzb3J0ZWQobCkgZm9yIGwgaW4gb3V0XQoKY2xhc3MgUGxhbm5lcjoKICAgICMgZGVjb21wb3NlIC0+IG9yZGVyIC0+IGRpc3BhdGNoIGVhY2ggdmlhIHRoZSBSb3V0ZXIgKHRocmVhZGluZyBkZXBlbmRlbmN5CiAgICAjIG91dHB1dHMgZm9yd2FyZCkgLT4gdmFsaWRhdGUgZWFjaCByZXN1bHQgYW5kIFJFUExBTiBmYWlsdXJlcyBvbiB0aGUKICAgICMgZ2VuZXJhbGlzdCAtPiBhc3NlbWJsZS4gVGhlIFJvdXRlciBhbnN3ZXJzICJ3aG8iOyB0aGUgUGxhbm5lciBhbnN3ZXJzCiAgICAjICJ3aGF0IGFyZSB0aGUgdGFza3MsIGluIHdoYXQgb3JkZXIsIGFuZCBkaWQgZWFjaCBvbmUgYWN0dWFsbHkgc3VjY2VlZCIuCiAgICBkZWYgX19pbml0X18oc2VsZiwgZGVjb21wb3NlOiBDYWxsYWJsZSwgcm91dGVyLAogICAgICAgICAgICAgICAgIHZhbGlkYXRlOiBPcHRpb25hbFtDYWxsYWJsZV0gPSBOb25lKToKICAgICAgICBzZWxmLmRlY29tcG9zZSA9IGRlY29tcG9zZSAgICAgICAgICAgICAgICMgZ29hbF90ZXh0IC0+IFBsYW4KICAgICAgICBzZWxmLnJvdXRlciA9IHJvdXRlcgogICAgICAgIHNlbGYudmFsaWRhdGUgPSB2YWxpZGF0ZSBvciAobGFtYmRhIHN0ZXAsIG91dDogb3V0LmdldCgiYW5zd2VyIikgaXMgbm90IE5vbmUpCgogICAgZGVmIHBsYW4oc2VsZiwgZ29hbF90ZXh0OiBzdHIpIC0+IFBsYW46CiAgICAgICAgcmV0dXJuIHNlbGYuZGVjb21wb3NlKGdvYWxfdGV4dCkKCiAgICBkZWYgZXhlY3V0ZShzZWxmLCBwbGFuOiBQbGFuKSAtPiBkaWN0OgogICAgICAgIG9yZGVyID0gdG9wb19vcmRlcihwbGFuLnN0ZXBzKSAgICAgICAgICAgIyByYWlzZXMgb24gYSBiYWQgcGxhbgogICAgICAgIGJ5X2lkID0gcGxhbi5ieV9pZCgpCiAgICAgICAgcmVzdWx0cywgcmVwbGFubmVkID0ge30sIFtdCiAgICAgICAgZm9yIHNpZCBpbiBvcmRlcjoKICAgICAgICAgICAgc3RlcCA9IGJ5X2lkW3NpZF0KICAgICAgICAgICAgdGFzayA9IGRpY3Qoc3RlcC5wYXlsb2FkKQogICAgICAgICAgICB0YXNrWyJ0ZXh0Il0gPSBzdGVwLnRleHQKICAgICAgICAgICAgIyBUSFJFQUQgZWFjaCBkZXBlbmRlbmN5J3MgYW5zd2VyIGludG8gdGhpcyB0YXNrJ3MgY29udGV4dC4KICAgICAgICAgICAgdGFza1siY29udGV4dCJdID0ge2Q6IHJlc3VsdHNbZF0uZ2V0KCJhbnN3ZXIiKSBmb3IgZCBpbiBzdGVwLmRlcHN9CiAgICAgICAgICAgIG91dCA9IHNlbGYucm91dGVyLmRpc3BhdGNoKHRhc2spCiAgICAgICAgICAgIGlmIG5vdCBzZWxmLnZhbGlkYXRlKHN0ZXAsIG91dCk6CiAgICAgICAgICAgICAgICAjIFJFUExBTjogd2hvZXZlciB0aGUgcm91dGVyIHBpY2tlZCBmYWlsZWQuIEVzY2FsYXRlIHRoaXMgb25lCiAgICAgICAgICAgICAgICAjIHN0ZXAgc3RyYWlnaHQgdG8gdGhlIGdlbmVyYWxpc3QgKGl0IGtlZXBzIHRoZSB0aHJlYWRlZCBjb250ZXh0KS4KICAgICAgICAgICAgICAgIGcgPSBzZWxmLnJvdXRlci5nZW5lcmFsaXN0CiAgICAgICAgICAgICAgICBtID0gZy5hY3QoTWVzc2FnZSgicGxhbm5lciIsIGcubmFtZSwgInRhc2siLCB0YXNrKSkKICAgICAgICAgICAgICAgIG91dCA9IHsiYW5zd2VyIjogbS5jb250ZW50LmdldCgiYW5zd2VyIiksCiAgICAgICAgICAgICAgICAgICAgICAgInRva2VucyI6IG91dC5nZXQoInRva2VucyIsIDApICsgbS5tZXRhLmdldCgidG9rZW5zIiwgMCksCiAgICAgICAgICAgICAgICAgICAgICAgInRyYWNlIjogb3V0LmdldCgidHJhY2UiLCBbXSkgKyBbZy5uYW1lXSwKICAgICAgICAgICAgICAgICAgICAgICAiZXNjYWxhdGVkIjogVHJ1ZSwgInJlYXNvbiI6ICJyZXBsYW4ifQogICAgICAgICAgICAgICAgcmVwbGFubmVkLmFwcGVuZChzaWQpCiAgICAgICAgICAgIHJlc3VsdHNbc2lkXSA9IG91dAogICAgICAgIHJldHVybiB7ImdvYWwiOiBwbGFuLmdvYWwsICJvcmRlciI6IG9yZGVyLCAicmVzdWx0cyI6IHJlc3VsdHMsCiAgICAgICAgICAgICAgICAicmVwbGFubmVkIjogcmVwbGFubmVkLAogICAgICAgICAgICAgICAgImFuc3dlcnMiOiB7azogdi5nZXQoImFuc3dlciIpIGZvciBrLCB2IGluIHJlc3VsdHMuaXRlbXMoKX0sCiAgICAgICAgICAgICAgICAidG9rZW5zIjogc3VtKHYuZ2V0KCJ0b2tlbnMiLCAwKSBmb3IgdiBpbiByZXN1bHRzLnZhbHVlcygpKX0K",
    "reliability.py": "IyBvcmNoZXN0cmEvcmVsaWFiaWxpdHkucHkgIC0tICB0aGUgbGF5ZXIgdGhhdCBtYWtlcyBvcmNoZXN0cmF0aW9uIHRydXN0d29ydGh5LgojIEw4MCdzIFBsYW5uZXIgZGlkIGEgU0lOR0xFLVNIT1QgcmVwbGFuIChvbmUgcmV0cnkgb250byB0aGUgZ2VuZXJhbGlzdCkuIFRoYXQKIyBpcyBub3QgcmVsaWFiaWxpdHkuIFJlbGlhYmlsaXR5IGlzIGEgZGlzY2lwbGluZWQgbGF5ZXI6IGRpc3Rpbmd1aXNoIGZhaWx1cmVzCiMgd29ydGggcmV0cnlpbmcgZnJvbSBvbmVzIHRoYXQgYXJlbid0LCByZXRyeSB0aGUgdHJhbnNpZW50IG9uZXMgd2l0aCBCQUNLT0ZGCiMgc28geW91IG5laXRoZXIgZ2l2ZSB1cCB0b28gZWFybHkgbm9yIGhhbW1lciBhIHN0cnVnZ2xpbmcgZGVwZW5kZW5jeSwgYm91bmQKIyBldmVyeSBjYWxsIHdpdGggYSBUSU1FT1VUIHNvIGEgaHVuZyBzdGVwIGNhbid0IHN0YWxsIHRoZSBwbGFuLCB0cmlwIGEgQ0lSQ1VJVAojIEJSRUFLRVIgc28gYSBwZXJzaXN0ZW50bHktYnJva2VuIGRlcGVuZGVuY3kgaXMgaXNvbGF0ZWQgaW5zdGVhZCBvZiByZXRyaWVkCiMgaW50byB0aGUgZ3JvdW5kLCBhbmQgZ2l2ZSBhIHBsYW4gYW4gZXhwbGljaXQgUEFSVElBTC1GQUlMVVJFIHBvbGljeSBzbwojICIyIG9mIDUgc3RlcHMgZmFpbGVkIiByZXNvbHZlcyB0byBhIGRlZmluZWQgZGVncmFkZWQgcmVzdWx0LCBub3QgYSBjcmFzaC4KaW1wb3J0IHRpbWUsIHJhbmRvbQppbXBvcnQgY29uY3VycmVudC5mdXR1cmVzIGFzIF9jZgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZSwgT3B0aW9uYWwsIExpc3QsIERpY3QsIEFueQoKIyAtLS0tIEZhaWx1cmUgdGF4b25vbXk6IHRoZSBzaW5nbGUgbW9zdCBpbXBvcnRhbnQgZGlzdGluY3Rpb24gLS0tLS0tLS0tLS0tLS0tLS0KY2xhc3MgVHJhbnNpZW50KEV4Y2VwdGlvbik6CiAgICAiIiJXb3J0aCByZXRyeWluZzogdGltZW91dCwgcmF0ZS1saW1pdCwgNXh4LCBjb25uZWN0aW9uIHJlc2V0LCBibGlwLiIiIgpjbGFzcyBQZXJtYW5lbnQoRXhjZXB0aW9uKToKICAgICIiIk5PVCB3b3J0aCByZXRyeWluZzogYmFkIGlucHV0LCBhdXRoLCA0eHgsIGEgdmFsaWRhdGlvbiBlcnJvci4iIiIKY2xhc3MgVGltZW91dChUcmFuc2llbnQpOgogICAgIiIiQSBjYWxsIHRoYXQgb3ZlcnJhbiBpdHMgZGVhZGxpbmUuIFRyYW5zaWVudCBieSBuYXR1cmUuIiIiCmNsYXNzIENpcmN1aXRPcGVuKFRyYW5zaWVudCk6CiAgICAiIiJUaGUgYnJlYWtlciBpcyBvcGVuOyB0aGUgY2FsbCB3YXMgcmVqZWN0ZWQgd2l0aG91dCB0b3VjaGluZyB0aGUgYmFja2VuZC4iIiIKCiMgLS0tLSBUaW1lb3V0OiBib3VuZCB0aGUgQ0FMTEVSIGV2ZW4gaWYgdGhlIGNhbGxlZSB3b24ndCBjb29wZXJhdGUgLS0tLS0tLS0tLS0tCmRlZiBjYWxsX3dpdGhfdGltZW91dChmbiwgdGltZW91dF9zLCAqYSwgKiprKToKICAgICMgUHVyZSBQeXRob24gY2FuJ3Qgc2FmZWx5IGtpbGwgYSB0aHJlYWQsIHNvIHdlIHJ1biBmbiBpbiBhIHdvcmtlciBhbmQKICAgICMgYWJhbmRvbiBpdHMgcmVzdWx0IGlmIGl0IG92ZXJydW5zLiBUaGlzIGJvdW5kcyB0aGUgY2FsbGVyJ3MgbGF0ZW5jeSwKICAgICMgd2hpY2ggaXMgd2hhdCBhIGRlYWRsaW5lIGlzIGZvci4KICAgIHdpdGggX2NmLlRocmVhZFBvb2xFeGVjdXRvcihtYXhfd29ya2Vycz0xKSBhcyBleDoKICAgICAgICBmdXQgPSBleC5zdWJtaXQoZm4sICphLCAqKmspCiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXR1cm4gZnV0LnJlc3VsdCh0aW1lb3V0PXRpbWVvdXRfcykKICAgICAgICBleGNlcHQgX2NmLlRpbWVvdXRFcnJvcjoKICAgICAgICAgICAgcmFpc2UgVGltZW91dCgiY2FsbCBleGNlZWRlZCAlLjNmcyIgJSB0aW1lb3V0X3MpCgojIC0tLS0gUmV0cnkgd2l0aCBleHBvbmVudGlhbCBiYWNrb2ZmICsgaml0dGVyIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpAZGF0YWNsYXNzCmNsYXNzIFJldHJ5UG9saWN5OgogICAgbWF4X2F0dGVtcHRzOiBpbnQgPSAzCiAgICBiYXNlX2RlbGF5OiBmbG9hdCA9IDAuMDUKICAgIGZhY3RvcjogZmxvYXQgPSAyLjAKICAgIG1heF9kZWxheTogZmxvYXQgPSAyLjAKICAgIGppdHRlcjogZmxvYXQgPSAwLjUgICAgICAgICAgICAjIGZyYWN0aW9uIG9mIGVhY2ggZGVsYXkgdGhhdCBpcyByYW5kb21pemVkCiAgICBkZWYgZGVsYXkoc2VsZiwgYXR0ZW1wdDogaW50LCBybmc9cmFuZG9tKSAtPiBmbG9hdDoKICAgICAgICAjIGF0dGVtcHQgaXMgMS1iYXNlZDogdGhlIHdhaXQgQUZURVIgYXR0ZW1wdCBrLCBiZWZvcmUgYXR0ZW1wdCBrKzEuCiAgICAgICAgcmF3ID0gbWluKHNlbGYubWF4X2RlbGF5LCBzZWxmLmJhc2VfZGVsYXkgKiAoc2VsZi5mYWN0b3IgKiogKGF0dGVtcHQgLSAxKSkpCiAgICAgICAgIyAiZnVsbCBqaXR0ZXIiIGluIFtyYXcqKDEtaml0dGVyKSwgcmF3XSBzcHJlYWRzIHJldHJpZXMgc28gTiBjbGllbnRzCiAgICAgICAgIyB0aGF0IGZhaWxlZCB0b2dldGhlciBkb24ndCBhbGwgcmV0cnkgaW4gbG9ja3N0ZXAgKHRodW5kZXJpbmcgaGVyZCkuCiAgICAgICAgcmV0dXJuIHJhdyAqICgxIC0gc2VsZi5qaXR0ZXIgKiBybmcucmFuZG9tKCkpCgpAZGF0YWNsYXNzCmNsYXNzIEF0dGVtcHQ6CiAgICBuOiBpbnQKICAgIG9rOiBib29sCiAgICBlcnJvcjogT3B0aW9uYWxbc3RyXQogICAgc2xlcHQ6IGZsb2F0CgpkZWYgcmV0cnkoZm4sIHBvbGljeTogUmV0cnlQb2xpY3ksIHNsZWVwPXRpbWUuc2xlZXAsIHJuZz1yYW5kb20pIC0+IGRpY3Q6CiAgICAjIFJldHJ5IE9OTFkgVHJhbnNpZW50IGZhaWx1cmVzLiBBIFBlcm1hbmVudCBmYWlsdXJlIGZhaWxzIEZBU1QgKG5vIHJldHJ5LAogICAgIyBubyBiYWNrb2ZmKSAtLSByZXRyeWluZyBhIGJhZCBpbnB1dCBqdXN0IHdhc3RlcyB0aW1lIGFuZCBtb25leS4KICAgIGF0dGVtcHRzOiBMaXN0W0F0dGVtcHRdID0gW10KICAgIGxhc3QgPSBOb25lCiAgICBmb3IgbiBpbiByYW5nZSgxLCBwb2xpY3kubWF4X2F0dGVtcHRzICsgMSk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICB2YWwgPSBmbigpCiAgICAgICAgICAgIGF0dGVtcHRzLmFwcGVuZChBdHRlbXB0KG4sIFRydWUsIE5vbmUsIDAuMCkpCiAgICAgICAgICAgIHJldHVybiB7Im9rIjogVHJ1ZSwgInZhbHVlIjogdmFsLCAiYXR0ZW1wdHMiOiBhdHRlbXB0cywgInJlYXNvbiI6ICJvayJ9CiAgICAgICAgZXhjZXB0IFBlcm1hbmVudCBhcyBlOgogICAgICAgICAgICBhdHRlbXB0cy5hcHBlbmQoQXR0ZW1wdChuLCBGYWxzZSwgInBlcm1hbmVudDolcyIgJSBlLCAwLjApKQogICAgICAgICAgICByZXR1cm4geyJvayI6IEZhbHNlLCAidmFsdWUiOiBOb25lLCAiYXR0ZW1wdHMiOiBhdHRlbXB0cywgInJlYXNvbiI6ICJwZXJtYW5lbnQifQogICAgICAgIGV4Y2VwdCBUcmFuc2llbnQgYXMgZToKICAgICAgICAgICAgbGFzdCA9IHN0cihlKQogICAgICAgICAgICBpZiBuID09IHBvbGljeS5tYXhfYXR0ZW1wdHM6CiAgICAgICAgICAgICAgICBhdHRlbXB0cy5hcHBlbmQoQXR0ZW1wdChuLCBGYWxzZSwgInRyYW5zaWVudDolcyIgJSBlLCAwLjApKQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgZCA9IHBvbGljeS5kZWxheShuLCBybmcpCiAgICAgICAgICAgIGF0dGVtcHRzLmFwcGVuZChBdHRlbXB0KG4sIEZhbHNlLCAidHJhbnNpZW50OiVzIiAlIGUsIGQpKQogICAgICAgICAgICBzbGVlcChkKQogICAgcmV0dXJuIHsib2siOiBGYWxzZSwgInZhbHVlIjogTm9uZSwgImF0dGVtcHRzIjogYXR0ZW1wdHMsICJyZWFzb24iOiAiZXhoYXVzdGVkIiwgImxhc3QiOiBsYXN0fQoKIyAtLS0tIENpcmN1aXQgYnJlYWtlcjogc3RvcCBoYW1tZXJpbmcgYSBkZXBlbmRlbmN5IHRoYXQgaXMgY2xlYXJseSBkb3duIC0tLS0tLS0tCkBkYXRhY2xhc3MKY2xhc3MgQ2lyY3VpdEJyZWFrZXI6CiAgICBmYWlsX3RocmVzaG9sZDogaW50ID0gMyAgICAgICAjIGNvbnNlY3V0aXZlIGZhaWx1cmVzIC0+IE9QRU4KICAgIHJlc2V0X3RpbWVvdXQ6IGZsb2F0ID0gMC4yICAgICMgc2Vjb25kcyB0byBzdGF5IE9QRU4gYmVmb3JlIGEgSEFMRl9PUEVOIHByb2JlCiAgICBoYWxmX29wZW5fc3VjY2Vzc2VzOiBpbnQgPSAxICAjIHN1Y2Nlc3NlcyBpbiBIQUxGX09QRU4gbmVlZGVkIHRvIENMT1NFCiAgICBub3c6IENhbGxhYmxlID0gdGltZS5tb25vdG9uaWMKICAgIHN0YXRlOiBzdHIgPSAiY2xvc2VkIgogICAgZmFpbHM6IGludCA9IDAKICAgIG9wZW5lZF9hdDogZmxvYXQgPSAwLjAKICAgIGhhbGZfb2s6IGludCA9IDAKICAgIGRlZiBhbGxvdyhzZWxmKSAtPiBib29sOgogICAgICAgICMgT1BFTiByZWplY3RzIGluc3RhbnRseSAobm8gYmFja2VuZCBjYWxsKS4gQWZ0ZXIgdGhlIGNvb2xkb3duIHdlIGFsbG93CiAgICAgICAgIyBPTkUgcHJvYmUgKEhBTEZfT1BFTikgdG8gdGVzdCB3aGV0aGVyIHRoZSBkZXBlbmRlbmN5IGhhcyByZWNvdmVyZWQuCiAgICAgICAgaWYgc2VsZi5zdGF0ZSA9PSAib3BlbiI6CiAgICAgICAgICAgIGlmIHNlbGYubm93KCkgLSBzZWxmLm9wZW5lZF9hdCA+PSBzZWxmLnJlc2V0X3RpbWVvdXQ6CiAgICAgICAgICAgICAgICBzZWxmLnN0YXRlLCBzZWxmLmhhbGZfb2sgPSAiaGFsZl9vcGVuIiwgMAogICAgICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgcmV0dXJuIFRydWUKICAgIGRlZiBvbl9zdWNjZXNzKHNlbGYpOgogICAgICAgIGlmIHNlbGYuc3RhdGUgPT0gImhhbGZfb3BlbiI6CiAgICAgICAgICAgIHNlbGYuaGFsZl9vayArPSAxCiAgICAgICAgICAgIGlmIHNlbGYuaGFsZl9vayA+PSBzZWxmLmhhbGZfb3Blbl9zdWNjZXNzZXM6CiAgICAgICAgICAgICAgICBzZWxmLnN0YXRlLCBzZWxmLmZhaWxzID0gImNsb3NlZCIsIDAKICAgICAgICBlbHNlOgogICAgICAgICAgICBzZWxmLmZhaWxzID0gMAogICAgZGVmIG9uX2ZhaWx1cmUoc2VsZik6CiAgICAgICAgaWYgc2VsZi5zdGF0ZSA9PSAiaGFsZl9vcGVuIjoKICAgICAgICAgICAgc2VsZi5zdGF0ZSwgc2VsZi5vcGVuZWRfYXQgPSAib3BlbiIsIHNlbGYubm93KCkgICAjIHByb2JlIGZhaWxlZCAtPiByZW9wZW4KICAgICAgICBlbHNlOgogICAgICAgICAgICBzZWxmLmZhaWxzICs9IDEKICAgICAgICAgICAgaWYgc2VsZi5mYWlscyA+PSBzZWxmLmZhaWxfdGhyZXNob2xkOgogICAgICAgICAgICAgICAgc2VsZi5zdGF0ZSwgc2VsZi5vcGVuZWRfYXQgPSAib3BlbiIsIHNlbGYubm93KCkKCiMgLS0tLSBSZWxpYWJsZTogY29tcG9zZSBicmVha2VyIChvdXRlcikgKyB0aW1lb3V0ICsgcmV0cnkgKGlubmVyKSAtLS0tLS0tLS0tLS0tCmNsYXNzIFJlbGlhYmxlOgogICAgIyBPbmUgY2FsbCgpID0gYnJlYWtlciBnYXRlIC0+ICh0aW1lb3V0LXdyYXBwZWQpIHJldHJ5IC0+IHJlcG9ydCB0aGUgRklOQUwKICAgICMgb3V0Y29tZSB0byB0aGUgYnJlYWtlci4gQnJlYWtlciBpcyB0aGUgT1VURVIgZ3VhcmQgc28gYW4gb3BlbiBjaXJjdWl0CiAgICAjIHJlamVjdHMgZmFzdCBXSVRIT1VUIHNwZW5kaW5nIHRoZSByZXRyeSBidWRnZXQgb24gYmFja29mZiBzbGVlcHMuCiAgICBkZWYgX19pbml0X18oc2VsZiwgcG9saWN5OiBSZXRyeVBvbGljeSwgYnJlYWtlcjogT3B0aW9uYWxbQ2lyY3VpdEJyZWFrZXJdID0gTm9uZSwKICAgICAgICAgICAgICAgICB0aW1lb3V0X3M6IE9wdGlvbmFsW2Zsb2F0XSA9IE5vbmUpOgogICAgICAgIHNlbGYucG9saWN5LCBzZWxmLmJyZWFrZXIsIHNlbGYudGltZW91dF9zID0gcG9saWN5LCBicmVha2VyLCB0aW1lb3V0X3MKICAgICAgICBzZWxmLnJlamVjdGVkID0gMAogICAgZGVmIGNhbGwoc2VsZiwgZm4sIHNsZWVwPXRpbWUuc2xlZXAsIHJuZz1yYW5kb20pIC0+IGRpY3Q6CiAgICAgICAgaWYgc2VsZi5icmVha2VyIGFuZCBub3Qgc2VsZi5icmVha2VyLmFsbG93KCk6CiAgICAgICAgICAgIHNlbGYucmVqZWN0ZWQgKz0gMQogICAgICAgICAgICByZXR1cm4geyJvayI6IEZhbHNlLCAidmFsdWUiOiBOb25lLCAiYXR0ZW1wdHMiOiBbXSwgInJlYXNvbiI6ICJjaXJjdWl0X29wZW4ifQogICAgICAgIHRhcmdldCA9IGZuIGlmIHNlbGYudGltZW91dF9zIGlzIE5vbmUgZWxzZSAobGFtYmRhOiBjYWxsX3dpdGhfdGltZW91dChmbiwgc2VsZi50aW1lb3V0X3MpKQogICAgICAgIHJlcyA9IHJldHJ5KHRhcmdldCwgc2VsZi5wb2xpY3ksIHNsZWVwPXNsZWVwLCBybmc9cm5nKQogICAgICAgIGlmIHNlbGYuYnJlYWtlcjoKICAgICAgICAgICAgc2VsZi5icmVha2VyLm9uX3N1Y2Nlc3MoKSBpZiByZXNbIm9rIl0gZWxzZSBzZWxmLmJyZWFrZXIub25fZmFpbHVyZSgpCiAgICAgICAgcmV0dXJuIHJlcwoKIyAtLS0tIFBhcnRpYWwtZmFpbHVyZSBwb2xpY3kgZm9yIGEgcGFyYWxsZWwgbGF5ZXIgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KZGVmIHJ1bl9sYXllcihzdGVwX2lkczogTGlzdFtzdHJdLCBydW5fb25lOiBDYWxsYWJsZVtbc3RyXSwgZGljdF0sCiAgICAgICAgICAgICAgcG9saWN5OiBzdHIgPSAiYWxsX29yX25vdGhpbmciLCBxdW9ydW06IE9wdGlvbmFsW2ludF0gPSBOb25lKSAtPiBkaWN0OgogICAgIyBydW5fb25lKHNpZCkgLT4geyJvayI6IGJvb2wsIC4uLn0uIERlY2lkZSBhIExBWUVSIHN0YXR1cyB3aGVuIHNvbWUgc3RlcHMKICAgICMgZmFpbCwgaW5zdGVhZCBvZiBsZXR0aW5nIG9uZSBmYWlsdXJlIGJlY29tZSBhbiB1bmhhbmRsZWQgZXhjZXB0aW9uLgogICAgcmVzdWx0cyA9IHtzaWQ6IHJ1bl9vbmUoc2lkKSBmb3Igc2lkIGluIHN0ZXBfaWRzfQogICAgb2sgID0gW3MgZm9yIHMgaW4gc3RlcF9pZHMgaWYgcmVzdWx0c1tzXVsib2siXV0KICAgIGJhZCA9IFtzIGZvciBzIGluIHN0ZXBfaWRzIGlmIG5vdCByZXN1bHRzW3NdWyJvayJdXQogICAgbiA9IGxlbihzdGVwX2lkcykKICAgIGlmIHBvbGljeSA9PSAiYWxsX29yX25vdGhpbmciOgogICAgICAgIHN0YXR1cyA9ICJvayIgaWYgbm90IGJhZCBlbHNlICJmYWlsZWQiCiAgICBlbGlmIHBvbGljeSA9PSAiYmVzdF9lZmZvcnQiOgogICAgICAgIHN0YXR1cyA9ICJvayIgaWYgbm90IGJhZCBlbHNlICJkZWdyYWRlZCIKICAgIGVsaWYgcG9saWN5ID09ICJxdW9ydW0iOgogICAgICAgIG5lZWQgPSBxdW9ydW0gaWYgcXVvcnVtIGlzIG5vdCBOb25lIGVsc2UgKG4gLy8gMiArIDEpCiAgICAgICAgc3RhdHVzID0gIm9rIiBpZiBsZW4ob2spID49IG5lZWQgZWxzZSAiZmFpbGVkIgogICAgZWxzZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ1bmtub3duIHBhcnRpYWwtZmFpbHVyZSBwb2xpY3kgJXIiICUgcG9saWN5KQogICAgcmV0dXJuIHsic3RhdHVzIjogc3RhdHVzLCAib2siOiBvaywgImZhaWxlZCI6IGJhZCwgInJlc3VsdHMiOiByZXN1bHRzLAogICAgICAgICAgICAicG9saWN5IjogcG9saWN5LCAibiI6IG59Cg==",
}
for fname, b64 in _MODULES.items():
    with open(os.path.join(BASE, "orchestra", fname), "wb") as f:
        f.write(base64.b64decode(b64))
# make it a package
open(os.path.join(BASE, "orchestra", "__init__.py"), "w").close()

if BASE not in sys.path:
    sys.path.insert(0, BASE)
importlib.invalidate_caches()

from orchestra.core import Message, Agent
from orchestra.router import Route, Classifier, Router
from orchestra.planner import Step, Plan, topo_order, layers, Planner
from orchestra.reliability import (
    Transient, Permanent, Timeout, CircuitOpen,
    RetryPolicy, retry, call_with_timeout,
    CircuitBreaker, Reliable, run_layer,
)
print("orchestra loaded:", [m for m in _MODULES])
print("reliability exports:", ["Transient","Permanent","RetryPolicy","retry",
      "call_with_timeout","CircuitBreaker","Reliable","run_layer"])

# One collector for every assertion in the lesson (checked at the end).
CHECKS = []
def check(name, cond):
    CHECKS.append((name, bool(cond)))
    print(("  PASS" if cond else "  FAIL"), "·", name)


## §1 · The one distinction that matters: transient vs permanent

Before you can retry intelligently you must answer one question about every
failure: **is it worth trying again?**

- **Transient** — a *blip*: a 503, a rate-limit, a dropped connection, a
  timeout. The exact same call, a moment later, would probably succeed.
  **Retry these.**
- **Permanent** — a *defect*: bad input, a validation error, a 401/404. The
  same call will fail the same way every time. **Do not retry** — you'd just
  burn time and money to get the same failure.

`orchestra.reliability` encodes this as an exception taxonomy:
`Transient` (and its subclasses `Timeout`, `CircuitOpen`) vs `Permanent`.
Everything downstream keys off it: `retry` retries `Transient`, fails fast on
`Permanent`.


In [ ]:
# A backend driven by a scripted list of outcomes lets us make every
# "flaky service" experiment fully deterministic.
def scripted(seq):
    """seq is a list like ["T","T","ok"] -> raise Transient twice, then return."""
    box = {"i": 0}
    def fn():
        o = seq[min(box["i"], len(seq) - 1)]; box["i"] += 1
        if o == "T": raise Transient("transient blip")
        if o == "P": raise Permanent("bad input")
        return "value-%d" % box["i"]
    return fn

# The naive approach — "just call it" — dies on the FIRST blip.
naive_died = False
try:
    scripted(["T", "ok"])()          # one transient, then it would have worked
except Transient:
    naive_died = True
check("a single call dies on the first transient blip", naive_died)

# It also can't tell a blip from a defect — both are just exceptions to it.
print("\nThe taxonomy lets the reliability layer treat these two DIFFERENTLY:")
print("  Transient -> retry;   Permanent -> fail fast.")


## §2 · Retry with exponential backoff + jitter

`retry(fn, policy)` calls `fn` up to `policy.max_attempts` times. It retries a
`Transient` failure and **fails fast on a `Permanent` one**. Between attempts it
waits — and *how* it waits matters:

- **Exponential backoff** — the wait grows (`base · factor^(k-1)`), capped at
  `max_delay`. A service that's overloaded gets *more* breathing room each
  retry, not a fixed drumbeat.
- **Jitter** — randomize each delay. If 500 clients all failed at the same
  instant and all retry after exactly 100 ms, they stampede the recovering
  service in lockstep (the **thundering herd**). "Full jitter" spreads them
  across the interval so the load smooths out.

💡 **EXPERIMENT:** set `jitter=0.0` and re-run the last check — the 100 delays
collapse to a single value and the herd guard disappears.


In [ ]:
import random
rng = random.Random(81)
pol = RetryPolicy(max_attempts=3, base_delay=0.01, jitter=0.5)
def nosleep(_): pass          # deterministic tests: don't actually wait

# (a) a transient that clears within the budget -> recovered
r = retry(scripted(["T", "T", "ok"]), pol, sleep=nosleep, rng=rng)
check("retry recovers a transient after 2 failed attempts", r["ok"] and len(r["attempts"]) == 3)

# (b) a permanent failure -> fails FAST: one attempt, no backoff, no retry
r = retry(scripted(["P", "ok"]), pol, sleep=nosleep, rng=rng)
check("permanent failure fails fast (1 attempt, reason='permanent')",
      (not r["ok"]) and r["reason"] == "permanent" and len(r["attempts"]) == 1)

# (c) a service that's down for longer than the budget -> give up, honestly
r = retry(scripted(["T", "T", "T", "ok"]), pol, sleep=nosleep, rng=rng)
check("retry exhausts after max_attempts (reason='exhausted')",
      (not r["ok"]) and r["reason"] == "exhausted" and len(r["attempts"]) == 3)

# (d) backoff really is exponential (measure the raw delays with jitter off)
p0 = RetryPolicy(base_delay=0.05, factor=2.0, jitter=0.0)
print("\nraw backoff:", [p0.delay(k) for k in (1, 2, 3)], "seconds")
check("backoff grows exponentially: 0.05 -> 0.10 -> 0.20",
      p0.delay(1) == 0.05 and p0.delay(2) == 0.10 and p0.delay(3) == 0.20)

# (e) jitter spreads 100 same-attempt delays across the interval (herd guard)
spread = {round(RetryPolicy(base_delay=0.1, jitter=0.5).delay(2, random.Random(i)), 6)
          for i in range(100)}
check("jitter spreads retries (thundering-herd guard)", len(spread) > 50)


## §3 · Timeouts: bound the caller even when the callee won't cooperate

A retry budget is useless if a single call *hangs forever* — you never reach the
retry. Every remote call needs a **deadline**.

`call_with_timeout(fn, timeout_s)` runs `fn` in a worker thread and raises
`Timeout` (a subclass of `Transient`) if it overruns. Pure Python can't safely
*kill* the runaway thread, so we **abandon** its result — but the caller's
latency is now bounded, which is the whole point of a deadline.

Because `Timeout` is `Transient`, it composes for free with retry: a slow
attempt times out, and the retry gives a *fresh* attempt a chance.


In [ ]:
import time
def slow():  time.sleep(0.20); return "late"
def fast():  return "quick"

# a deadline turns an unbounded hang into a bounded, catchable Timeout
fired = False
try:
    call_with_timeout(slow, 0.05)
except Timeout:
    fired = True
check("timeout fires on a call that overruns its deadline", fired)
check("a fast call returns normally under its deadline", call_with_timeout(fast, 0.5) == "quick")

# timeout + retry compose: first attempt hangs (times out), second is fast -> recovered
box = {"i": 0}
def slow_then_fast():
    box["i"] += 1
    if box["i"] == 1:
        time.sleep(0.20); return "late"      # first attempt overruns
    return "quick"                            # retry attempt is fast

rel = Reliable(RetryPolicy(max_attempts=2, base_delay=0.001, jitter=0.0), timeout_s=0.05)
res = rel.call(slow_then_fast, sleep=nosleep)
check("timeout + retry recover after one slow attempt", res["ok"] and res["value"] == "quick")


## §4 · Circuit breaker: stop hammering a dependency that's clearly down

Retry is right for a *blip*. But if a dependency is genuinely **down**, retrying
every request just piles load onto a service that's already struggling — and
makes *your* system slow too, since every request waits out its full backoff
budget before failing.

A **circuit breaker** fixes this. It watches a dependency and moves through
three states:

- **CLOSED** — healthy; calls pass through.
- **OPEN** — after `fail_threshold` consecutive failures, the breaker *trips*.
  Further calls are **rejected instantly** (no backend call, no waiting) for a
  cooldown period. This is the key move: fail fast instead of piling on.
- **HALF-OPEN** — after the cooldown, allow **one probe** through. If it
  succeeds, close the circuit (recovered); if it fails, re-open (still down).

`Reliable` composes the breaker as the **outer** guard around retry, so an open
circuit rejects *without* spending the retry budget on backoff sleeps.


In [ ]:
# A permanently-dead backend, counted so we can compare load.
pol4 = RetryPolicy(max_attempts=3, base_delay=0.001, jitter=0.0)

# WITHOUT a breaker: 10 requests each burn the full 3-attempt retry budget.
no_br = {"calls": 0}
def dead_A():
    no_br["calls"] += 1; raise Transient("service down")
for _ in range(10):
    retry(dead_A, pol4, sleep=nosleep)

# WITH a breaker: it trips after 3 consecutive failures and rejects the rest fast.
br = CircuitBreaker(fail_threshold=3, reset_timeout=999)   # no re-probe in this window
rel = Reliable(pol4, breaker=br)
with_br = {"calls": 0}
def dead_B():
    with_br["calls"] += 1; raise Transient("service down")
reasons = [rel.call(dead_B, sleep=nosleep)["reason"] for _ in range(10)]

print("backend calls WITHOUT breaker:", no_br["calls"], "  (10 requests x 3 attempts)")
print("backend calls WITH breaker:   ", with_br["calls"], "  (then it trips and rejects fast)")
print("rejections after trip:", reasons.count("circuit_open"), " · breaker state:", br.state)
check("without a breaker every request hammers the dead backend (30 calls)", no_br["calls"] == 30)
check("with a breaker, backend calls are bounded far below that", with_br["calls"] < no_br["calls"])
check("the breaker trips OPEN and rejects the rest fast",
      "circuit_open" in reasons and br.state == "open" and rel.rejected > 0)

# HALF-OPEN recovery: two failures trip it; after the cooldown one probe finds
# the backend healthy again and CLOSES the circuit.
br2 = CircuitBreaker(fail_threshold=2, reset_timeout=0.05, half_open_successes=1)
rel2 = Reliable(RetryPolicy(max_attempts=1, base_delay=0.001, jitter=0.0), breaker=br2)
seq = ["T", "T", "ok", "ok"]; bi = {"i": 0}
def flaky_then_ok():
    o = seq[min(bi["i"], len(seq) - 1)]; bi["i"] += 1
    if o == "T": raise Transient("down")
    return "ok"
rel2.call(flaky_then_ok, sleep=nosleep)      # fail 1
rel2.call(flaky_then_ok, sleep=nosleep)      # fail 2 -> OPEN
opened = br2.state == "open"
time.sleep(0.06)                             # wait out the cooldown
probe = rel2.call(flaky_then_ok, sleep=nosleep)   # HALF-OPEN probe -> healthy -> CLOSED
print("\nafter cooldown, probe ok:", probe["ok"], " · final state:", br2.state)
check("breaker opens after the failure threshold", opened)
check("half-open probe recovers the circuit to CLOSED", br2.state == "closed" and probe["ok"])


## §5 · Partial-failure policies: when 2 of 5 parallel steps fail

L80's `layers()` groups independent steps so they run in parallel. But in
production some of those steps *will* fail even after retries. If one raised
exception kills the whole layer, a single flaky sub-task takes down an otherwise
successful plan. You need an explicit **policy** for "some succeeded, some
didn't":

- **all-or-nothing** — any failure fails the layer (correct when the steps are
  a transaction: a half-done result is worse than none).
- **best-effort** — return what succeeded, mark the layer **degraded** (correct
  when steps are independent enrichments: 3 of 5 summaries is still useful).
- **quorum** — succeed if at least *k* of *n* passed (correct for redundancy:
  ask 5 replicas, accept the answer if 3 agree).

`run_layer(step_ids, run_one, policy=...)` runs each step and resolves the layer
to a defined **status** instead of letting one failure become an unhandled
exception.


In [ ]:
# A layer of 5 steps where 2 fail (d, e) after their own retries are exhausted.
outcomes = {"a": True, "b": True, "c": True, "d": False, "e": False}
def run_one(sid):
    return {"ok": outcomes[sid], "value": sid}
ids = list(outcomes)

aon = run_layer(ids, run_one, policy="all_or_nothing")
be  = run_layer(ids, run_one, policy="best_effort")
q3  = run_layer(ids, run_one, policy="quorum", quorum=3)
q4  = run_layer(ids, run_one, policy="quorum", quorum=4)

for lay in (aon, be, q3, q4):
    print("%-16s -> %-8s  ok=%s failed=%s" % (lay["policy"], lay["status"], lay["ok"], lay["failed"]))

check("all-or-nothing: any failure -> 'failed'", aon["status"] == "failed")
check("best-effort: partial success -> 'degraded' (keeps the 3 that worked)",
      be["status"] == "degraded" and be["ok"] == ["a", "b", "c"])
check("quorum(3 of 5): 3 succeeded -> 'ok'", q3["status"] == "ok")
check("quorum(4 of 5): only 3 succeeded -> 'failed'", q4["status"] == "failed")


## §6 · The payoff: a `ResilientPlanner` over the L80 plan

Now we wire it all together. We keep L80's decompose → order → thread-dependencies
machinery, but execute each step through the **reliability layer** and resolve
each parallel layer with a **partial-failure policy**.

The world: a `math-bot` that computes a total, and a **flaky** `bill-bot` whose
billing service returns a `503` the *first* time each step calls it, then
succeeds — a textbook transient blip. Billing depends on the math total
(threaded via `context`, exactly as in L80).

Watch the contrast:

- **Naive L80 `Planner.execute`** — the flaky billing agent raises on its first
  call, the exception propagates, and the **whole plan crashes**.
- **`ResilientPlanner`** — the retry absorbs the blip, the dependency is
  threaded correctly, and the plan **completes** with the billed amount equal to
  the computed total. No replan even needed — the blip never escaped the
  reliability layer.


In [ ]:
import re
# ── the world ──────────────────────────────────────────────────────────────
def math_backend(name, task):
    t = task["text"].lower()
    st = re.search(r"subtotal (\d+(?:\.\d+)?)", t)
    tx = re.search(r"tax (\d+(?:\.\d+)?)", t)
    if st and tx:
        return ({"answer": {"total": round(float(st.group(1)) * (1 + float(tx.group(1))), 2)}}, 20)
    return ({"answer": None}, 20)

class FlakyBilling:
    """503 on the first call for a given step, then succeeds. A real blip."""
    def __init__(self): self.seen = {}
    def backend(self, name, task):
        k = task["text"]; self.seen[k] = self.seen.get(k, 0) + 1
        if self.seen[k] == 1:
            raise Transient("billing service 503")
        amount = None
        for v in (task.get("context") or {}).values():
            if isinstance(v, dict) and "total" in v: amount = v["total"]
        if amount is None: return ({"answer": None}, 25)
        return ({"answer": {"invoice": "Billed $%.2f" % amount, "amount": amount}}, 25)

def gen_backend(name, task):
    return ({"answer": {"note": "generalist fallback"}}, 60)

flaky = FlakyBilling()
math_agent = Agent("math-bot", "math", math_backend)
bill_agent = Agent("bill-bot", "billing", flaky.backend)
generalist = Agent("generalist", "general", gen_backend)
clf = Classifier({"math": ["compute", "total", "subtotal", "tax"],
                  "billing": ["bill", "invoice", "customer"]})
router = Router(clf, {"math": math_agent, "billing": bill_agent}, generalist, threshold=0.6)

def decompose(goal):
    return Plan(goal, [
        Step("s0", "compute total subtotal 100 tax 0.1", [], {"category": "math"}),
        Step("s1", "bill customer acme for the invoice", ["s0"], {"category": "billing"}),
    ])

# ── naive L80 planner: crashes on the blip ──────────────────────────────────
flaky.seen.clear()
naive = Planner(decompose, router)
naive_crashed = False
try:
    naive.execute(naive.plan("compute the total then bill the customer"))
except Transient:
    naive_crashed = True
check("naive L80 execute() CRASHES on a transient billing blip", naive_crashed)


In [ ]:
# ── the ResilientPlanner: L80 structure + the reliability layer ─────────────
class ResilientPlanner:
    """Decompose+order+thread (L80), but run each step through Reliable.call
    and resolve each parallel layer with a partial-failure policy."""
    def __init__(self, decompose, router, reliable_factory, validate=None):
        self.decompose = decompose
        self.router = router
        self.reliable_factory = reliable_factory          # () -> a fresh Reliable
        self.validate = validate or (lambda step, out: out.get("answer") is not None)

    def plan(self, goal): return self.decompose(goal)

    def execute(self, plan, policy="all_or_nothing"):
        by_id = plan.by_id()
        results, replanned, layer_status = {}, [], {}
        for layer in layers(plan.steps):                  # dependency-ordered layers
            def run_one(sid):
                step = by_id[sid]
                task = dict(step.payload); task["text"] = step.text
                task["context"] = {d: results[d].get("answer") for d in step.deps}
                rel = self.reliable_factory()
                r = rel.call(lambda: self.router.dispatch(task))   # retry/timeout/breaker
                if r["ok"] and self.validate(step, r["value"]):
                    results[sid] = r["value"]; return {"ok": True, "sid": sid}
                # retries couldn't save it -> L80-style replan onto the generalist
                g = self.router.generalist
                m = g.act(Message("planner", g.name, "task", task))
                out = {"answer": m.content.get("answer"), "reason": "replan"}
                results[sid] = out; replanned.append(sid)
                return {"ok": self.validate(step, out), "sid": sid}
            layer_status[tuple(layer)] = run_layer(layer, run_one, policy=policy)["status"]
        return {"goal": plan.goal, "results": results, "replanned": replanned,
                "answers": {k: v.get("answer") for k, v in results.items()},
                "layer_status": layer_status}

flaky.seen.clear()
resilient = ResilientPlanner(
    decompose, router,
    reliable_factory=lambda: Reliable(RetryPolicy(max_attempts=3, base_delay=0.001, jitter=0.0)))
out = resilient.execute(resilient.plan("compute the total then bill the customer"))

print("answers:", out["answers"])
print("replanned:", out["replanned"], " · layer status:", list(out["layer_status"].values()))
check("resilient planner COMPLETES despite the blip", out["answers"]["s1"] is not None)
check("dependency threaded: billed amount == the computed total (110.0)",
      out["answers"]["s1"]["amount"] == 110.0)
check("the retry absorbed the blip, so no replan was needed", out["replanned"] == [])


In [ ]:
# ── and when a dependency is PERMANENTLY down: retry can't help, so the step
#    is replanned onto the generalist, and best-effort keeps the plan's other
#    results instead of crashing the whole thing. ─────────────────────────────
class DeadBilling:
    def backend(self, name, task): raise Transient("billing hard-down")

dead_router = Router(clf,
    {"math": math_agent, "billing": Agent("bill-bot", "billing", DeadBilling().backend)},
    generalist, threshold=0.6)
resilient2 = ResilientPlanner(
    decompose, dead_router,
    reliable_factory=lambda: Reliable(RetryPolicy(max_attempts=2, base_delay=0.001, jitter=0.0)))
out2 = resilient2.execute(resilient2.plan("compute the total then bill the customer"),
                          policy="best_effort")

print("answers:", out2["answers"], " · replanned:", out2["replanned"])
check("a permanently-broken step is replanned, not crashed", "s1" in out2["replanned"])
check("partial success preserved: the math step still holds its answer (110.0)",
      out2["answers"]["s0"]["total"] == 110.0)


## §7 · Ten reliability pitfalls

| # | Pitfall | Why it bites | Fix |
|---|---|---|---|
| 1 | **Retrying a permanent failure** | burns time/money to get the same 400 | classify first; retry `Transient` only |
| 2 | **No backoff** (fixed-interval retry) | a fixed drumbeat keeps overloaded services overloaded | exponential backoff |
| 3 | **No jitter** | synchronized clients stampede on recovery (thundering herd) | full jitter |
| 4 | **Unbounded retries** | one bad request retries forever, starving others | cap `max_attempts` |
| 5 | **No timeout** | a hung call stalls the plan; you never reach the retry | a deadline on every call |
| 6 | **No circuit breaker** | you keep hammering a service that's already down | trip after N fails, reject fast |
| 7 | **Breaker never re-probes** | it stays open after the service recovers | half-open probe after a cooldown |
| 8 | **One failure crashes the layer** | a single flaky step kills a good plan | an explicit partial-failure policy |
| 9 | **Wrong policy for the job** | best-effort on a transaction ships a half-done result | all-or-nothing vs best-effort vs quorum |
| 10 | **Silent degradation** | callers can't tell "ok" from "degraded" | return an explicit layer **status** |

💡 Pitfall 10 connects straight back to Phase 8: emit one **L72 trace span** per
step with its final status + attempt count, and an **L73 alert** when the
degraded-layer rate crosses your SLO. Reliability you can't *observe* isn't
reliability you can trust.


## §8 · Ship `orchestra/reliability.py`

The module is already written to disk by the setup cell (that's the single
source of truth every cell above imported from). Here we reload it on a fresh
import and run three independent smoke tests to confirm the *shipped artifact* —
not just the in-notebook demos — behaves.


In [ ]:
import importlib, orchestra.reliability as _rel
importlib.reload(_rel)

# smoke 1: retry fails fast on Permanent
s1 = _rel.retry(lambda: (_ for _ in ()).throw(_rel.Permanent("x")),
                _rel.RetryPolicy(max_attempts=5), sleep=lambda _: None)
check("[ship] retry fails fast on Permanent", s1["reason"] == "permanent" and len(s1["attempts"]) == 1)

# smoke 2: a fresh breaker trips OPEN after its threshold
b = _rel.CircuitBreaker(fail_threshold=2)
b.on_failure(); mid = b.state; b.on_failure()
check("[ship] breaker: closed until threshold, then open", mid == "closed" and b.state == "open")

# smoke 3: run_layer resolves partial failure to a defined status
lay = _rel.run_layer(["a", "b"], lambda s: {"ok": s == "a"}, policy="best_effort")
check("[ship] run_layer marks a partial layer 'degraded'",
      lay["status"] == "degraded" and lay["ok"] == ["a"] and lay["failed"] == ["b"])

print("\norchestra/reliability.py verified as a shippable module.")


## ✅ Verification

Every `check(...)` in the lesson is collected here. The cell asserts they all
passed, so a clean run is proof the whole reliability story holds together.


In [ ]:
passed = sum(1 for _, ok in CHECKS if ok)
print("Verification —", passed, "/", len(CHECKS), "checks passed\n")
for name, ok in CHECKS:
    print(("  PASS" if ok else "  FAIL"), "·", name)
assert passed == len(CHECKS), "some checks failed — see FAILs above"
print("\nALL", len(CHECKS), "CHECKS PASSED ✅")


## Summary, homework & what's next

**What you built today**

| Concept | Primitive | The one-line idea |
|---|---|---|
| Failure taxonomy | `Transient` / `Permanent` | retry blips, never retry defects |
| Retry + backoff + jitter | `RetryPolicy`, `retry` | grow the wait, randomize it, cap the attempts |
| Timeout / deadline | `call_with_timeout` | bound the caller even if the callee hangs |
| Circuit breaker | `CircuitBreaker` | isolate a down dependency; fail fast, re-probe |
| Composed call | `Reliable` | breaker(outer) → timeout → retry(inner) |
| Partial-failure policy | `run_layer` | all-or-nothing vs best-effort vs quorum |
| Resilient orchestration | `ResilientPlanner` | L80 plan, now crash-proof and degradable |

**The through-line:** L80's Planner assumed every step *works*. L81 removes that
assumption. Retry handles the blip, backoff+jitter handles the herd, the timeout
handles the hang, the breaker handles the outage, and the partial-failure policy
handles "some of it failed" — turning a plan that *crashes on the first 503* into
one that *completes, degrades gracefully, or fails on defined terms*.

**Homework**

1. **Real LLM call:** wrap an actual `anthropic` call in `Reliable`, mapping
   `RateLimitError`/`APIStatusError(5xx)` → `Transient` and `BadRequestError` →
   `Permanent`. Confirm a rate-limit gets retried and a bad request fails fast.
2. **Budgeted retry:** add a wall-clock `deadline` to `retry` so it stops once
   total elapsed time (calls + backoff) exceeds a budget, even below
   `max_attempts`.
3. **Per-category breakers:** give the `Router` one `CircuitBreaker` per
   specialty so a down billing service never trips the healthy math one.
4. **Observability:** emit one **L72** span per step (`status`, `attempts`,
   `breaker_state`) and an **L73** alert when the degraded-layer rate crosses an
   SLO — reliability you can watch.
5. **`execute_parallel`:** fold L80's `layers()` + a `ThreadPoolExecutor` into
   `ResilientPlanner` so a layer's steps run concurrently, each with its own
   `Reliable.call`, aggregated by the partial-failure policy.

**Next — Lesson 82 · Phase-9 Capstone:** consolidate `orchestra`
(`core` + `blackboard` + `router` + `planner` + `reliability`) into one
installable, tested package and **ship it as your 4th open-source artifact** —
the multi-agent orchestration framework that ties Phase 9 together, alongside
`paper-distiller`, `agent-bench`, and `agent-obs`.
